In [1]:
# PACKAGES
## data manipulation
import pandas as pd
import numpy as np

## confidence interval
import math

# ANOVA
import scipy.stats as stats

## data viz
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
# PATHS AND METHODS

## My min max scaler
def my_scaler(x_unscaled, x_min=None, x_max=None):
    if x_min == None:
        x_min=x_unscaled.min()
    if x_max == None:
        x_max=x_unscaled.max()
    x_scaled = (x_unscaled - x_min)/(x_max - x_min)
    return x_scaled

## confidence interval 95%
def my_ci_95(stats, r=2):
    
    ci95_hi = []
    ci95_lo = []
    alpha = []

    for i in stats.index:
        m, c, s = stats.loc[i]
        a = 1.96*s/math.sqrt(c)
        ci95_hi.append(m + a)
        ci95_lo.append(m - a)
        alpha.append(a)

    stats['ci95_lo'] = ci95_lo
    stats['ci95_hi'] = ci95_hi
    stats['alpha'] = alpha

    stats['mean'] = round(stats['mean'], r)
    stats['alpha'] = round(stats['alpha'], r)

    return(stats.reset_index())

In [3]:
# DEFINING EXPERIMENT PARAMETERS
## seed list
seed_list = [428956419, 1954324947, 1145661099, 1835732737, 794161987, 1329531353, 200496737, 633816299, 1410143363, 1282538739]

## scenarion list
scenario_list = ['2ec_low_on', '2ec_low_off', '2ec_high_on', '2ec_high_off']
#scenario_list = ['2ec_high_off']

## algorithms list
#algo_list = ['tetris', 'thea', 'argos', 'faticanti']
algo_list = ['tetris', 'thea']

In [4]:
# IMPORT DATASETS

# dataframes
total_seed_server = pd.DataFrame()
total_seed_user = pd.DataFrame()
total_seed_service = pd.DataFrame()
# object to count the quantity of experiments
n_exp = 0

# loop to get results
for algo_name in algo_list:

    for scenario in scenario_list:

        ## count experiment
        n_exp = n_exp + 1

        for seed_value in seed_list:

            # server
            ## getting results by seeds
            each_seed_server = pd.read_csv(f"results/{algo_name}_logsserver_{scenario}_{seed_value}.csv")
            each_seed_server['seed'] = seed_value
            each_seed_server['scenario'] = scenario
            each_seed_server['algorithm'] = algo_name
            ## save results of all seeds for each experiment
            total_seed_server = pd.concat([total_seed_server, each_seed_server])
            total_seed_server = total_seed_server.reset_index(drop=True)

            # user
            ## getting results by seeds
            each_seed_user = pd.read_csv(f"results/{algo_name}_logsuser_{scenario}_{seed_value}.csv")
            each_seed_user['seed'] = seed_value
            each_seed_user['scenario'] = scenario
            each_seed_user['algorithm'] = algo_name
            ## save results of all seeds for each experiment
            total_seed_user = pd.concat([total_seed_user, each_seed_user])
            total_seed_user = total_seed_user.reset_index(drop=True)

            # service
            ## getting results by seeds
            each_seed_service = pd.read_csv(f"results/{algo_name}_logsservice_{scenario}_{seed_value}.csv")
            each_seed_service['seed'] = seed_value
            each_seed_service['scenario'] = scenario
            each_seed_service['algorithm'] = algo_name
            ## save results of all seeds for each experiment
            total_seed_service = pd.concat([total_seed_service, each_seed_service])
            total_seed_service = total_seed_service.reset_index(drop=True)

In [5]:
# FEATURE ENGINEERING

## create new obj to do feature engineering
dt_power = total_seed_server.loc[(total_seed_server['Available'])].reset_index(drop=True).copy()
dt_server = total_seed_server.loc[(total_seed_server['Available']) & (total_seed_server['CPU Demand'] > 0)].reset_index(drop=True).copy()
dt_user = total_seed_user.copy()
dt_service = total_seed_service.copy()

## alogrithm + scenario column
dt_power['algorithm_scenario'] = dt_power['algorithm'] + '_' + dt_server['scenario']
dt_server['algorithm_scenario'] = dt_server['algorithm'] + '_' + dt_server['scenario']
dt_user['algorithm_scenario'] = dt_user['algorithm'] + '_' + dt_user['scenario']
dt_service['algorithm_scenario'] = dt_service['algorithm'] + '_' + dt_service['scenario']

## Infra
dt_power['cloud'] = dt_power['scenario'].str.split('_', expand=True)[2]
dt_server['cloud'] = dt_server['scenario'].str.split('_', expand=True)[2]
dt_user['cloud'] = dt_user['scenario'].str.split('_', expand=True)[2]
dt_service['cloud'] = dt_service['scenario'].str.split('_', expand=True)[2]

## user dataset
# compute sla violation variable
dt_user['sla_violations'] = 0
dt_user.loc[dt_user['Delay sla deadlines'] < 0, 'sla_violations'] = 1
dt_user['cumulative_sla_violations'] = dt_user.groupby(['algorithm', 'scenario', 'seed'])['sla_violations'].cumsum()

# compute drops
dt_user_drop = dt_user.groupby(by=['algorithm', 'scenario', 'Object','seed'])['Delays'].agg(['last']).reset_index().rename(columns={'last':'Delays'})
dt_user_drop['drop'] = 0
dt_user_drop.loc[dt_user_drop['Delays'].isna(), 'drop'] = 1
## infra
dt_user_drop['cloud'] = dt_user_drop['scenario'].str.split('_', expand=True)[2]

## server dataset
# compute server access
dt_server['server_access'] = 0
dt_server.loc[dt_server['CPU Demand'] > 0, 'server_access'] = 1
dt_server['cumulative_server_access'] = dt_server.groupby(['Instance ID', 'algorithm', 'scenario', 'seed'])['server_access'].cumsum()
dt_total_server_access = dt_server.groupby(['algorithm', 'scenario', 'seed'])['server_access'].sum().reset_index().rename(columns={'server_access':'total_server_access'})
dt_server = dt_server.merge(dt_total_server_access, how='left', on=['algorithm', 'scenario', 'seed'])
dt_total_server_access_avg = dt_server.groupby(by=['algorithm', 'Object', 'scenario', 'seed'])['total_server_access'].agg(['last']).reset_index().groupby(by=['algorithm', 'scenario', 'Object'])['last'].agg(['mean', 'count']).reset_index()
dt_total_server_access_avg['product'] = dt_total_server_access_avg['mean']*dt_total_server_access_avg['count']
dt_total_server_access_avg = dt_total_server_access_avg.merge(dt_total_server_access_avg.groupby(by=['algorithm', 'scenario'])['product'].agg(['sum']).reset_index(), how='left', on=['algorithm', 'scenario'])
dt_total_server_access_avg['weight'] = dt_total_server_access_avg['product']/dt_total_server_access_avg['sum']
dt_total_server_access_avg['n_server_access'] = dt_total_server_access_avg['mean']*dt_total_server_access_avg['weight']
dt_total_server_access_avg = dt_total_server_access_avg.groupby(by=['algorithm', 'scenario'])['n_server_access'].agg(['sum']).reset_index().rename(columns={'sum':'n_server_access'})
dt_server_access_avg = dt_server.groupby(by=['algorithm', 'Object', 'scenario', 'seed'])['cumulative_server_access'].agg(['last']).reset_index().groupby(by=['algorithm', 'scenario', 'Object'])['last'].agg(['mean', 'count']).reset_index()
dt_server_access_avg['product'] = dt_server_access_avg['mean']*dt_server_access_avg['count']
dt_server_access_avg = dt_server_access_avg.merge(dt_server_access_avg.groupby(by=['algorithm', 'scenario'])['product'].agg(['sum']).reset_index(), how='left', on=['algorithm', 'scenario'])
dt_server_access_avg['weight'] = dt_server_access_avg['product']/dt_server_access_avg['sum']
dt_server_access_avg = dt_server_access_avg.merge(dt_total_server_access_avg, how='left', on=['algorithm', 'scenario'])
dt_server_access_avg['server_access_by_server'] = dt_server_access_avg['n_server_access']*dt_server_access_avg['weight']
## infra
dt_server_access_avg['cloud'] = dt_server_access_avg['scenario'].str.split('_', expand=True)[2]


# compute free resources
dt_server['CPU Free'] = dt_server['CPU'] - dt_server['CPU Demand']
dt_server['RAM Free'] = dt_server['RAM'] - dt_server['RAM Demand']
# compute usage metrics
dt_server['CPU Usage'] = (dt_server['CPU Demand'] * 100) / dt_server['CPU']
dt_server['RAM Usage'] = (dt_server['RAM Demand'] * 100) / dt_server['RAM']
dt_server['Servers Usage'] = (dt_server['CPU Usage'] + dt_server['RAM Usage']) / 2

# create new obj withou cloud server
dt_server_edge = dt_server.loc[(dt_server['Instance ID'] != 7)].reset_index(drop=True)

# normalization
## Imbalance
dt_server_edge_cpu = dt_server_edge.groupby(by=['algorithm', 'scenario'])[['CPU Free']].apply(my_scaler).reset_index().rename(columns={'CPU Free':'CPU Free_norm'})
dt_server_edge = dt_server_edge.merge(dt_server_edge_cpu[['CPU Free_norm', 'level_2']], how='left', left_index=True, right_on='level_2').drop(columns='level_2')
dt_server_edge_ram = dt_server_edge.groupby(by=['algorithm', 'scenario'])[['RAM Free']].apply(my_scaler).reset_index().rename(columns={'RAM Free':'RAM Free_norm'})
dt_server_edge = dt_server_edge.merge(dt_server_edge_ram[['RAM Free_norm', 'level_2']], how='left', left_index=True, right_on='level_2').drop(columns='level_2')
## Fragmentation
dt_server_edge_rf_cpu = dt_server_edge.groupby(by=['algorithm', 'scenario'])[['rf_cpu']].apply(my_scaler).reset_index().rename(columns={'rf_cpu':'rf_cpu_norm'})
dt_server_edge = dt_server_edge.merge(dt_server_edge_rf_cpu[['rf_cpu_norm', 'level_2']], how='left', left_index=True, right_on='level_2').drop(columns='level_2')
dt_server_edge_rf_ram = dt_server_edge.groupby(by=['algorithm', 'scenario'])[['rf_memory']].apply(my_scaler).reset_index().rename(columns={'rf_memory':'rf_memory_norm'})
dt_server_edge = dt_server_edge.merge(dt_server_edge_rf_ram[['rf_memory_norm', 'level_2']], how='left', left_index=True, right_on='level_2').drop(columns='level_2')

# compute Resource Imbalance Index (RII)
dt_server_edge['RII'] = dt_server_edge[['CPU Free_norm', 'RAM Free_norm']].std(axis=1)
# compute Resource Fragmentation Index (RFI)
dt_server_edge['RFI'] = dt_server_edge[['rf_cpu_norm', 'rf_memory_norm']].mean(axis=1)

In [6]:
# RESULTS TO USE ON OVERLEAF PLOTS

## SLA Violations
stats_sla = dt_user.groupby(by=['algorithm', 'scenario', 'seed'])['cumulative_sla_violations'].max().groupby(by=['algorithm', 'scenario']).agg(['mean', 'count', 'std'])
stats_sla = my_ci_95(stats_sla)
print(">>>>>>>>>>>>>>>>>>>> cumulative_sla_violations <<<<<<<<<<<<<<<<<<<<\n", stats_sla[['algorithm', 'scenario', 'mean', 'alpha']], '\n')

## Drop
stats_drop = dt_user_drop.groupby(by=['algorithm', 'scenario', 'seed'])['drop'].sum().groupby(by=['algorithm', 'scenario']).agg(['mean', 'count', 'std'])
stats_drop = my_ci_95(stats_drop)
print(">>>>>>>>>>>>>>>>>>>> drop <<<<<<<<<<<<<<<<<<<<\n", stats_drop[['algorithm', 'scenario', 'mean', 'alpha']], '\n')

## Delay
stats_delay = dt_user.groupby(by=['algorithm', 'scenario'])['Delays'].agg(['mean', 'count', 'std'])
stats_delay = my_ci_95(stats_delay)
print(">>>>>>>>>>>>>>>>>>>> Delays <<<<<<<<<<<<<<<<<<<<\n", stats_delay[['algorithm', 'scenario', 'mean', 'alpha']], '\n')

## Power Consumption
stats_power = dt_power.groupby(by=['algorithm', 'scenario'])['Power Consumption'].agg(['mean', 'count', 'std'])
stats_power = my_ci_95(stats_power)
print(">>>>>>>>>>>>>>>>>>>> Power <<<<<<<<<<<<<<<<<<<<\n", stats_power[['algorithm', 'scenario', 'mean', 'alpha']], '\n')

## RFI
stats_rfi = dt_server_edge.groupby(by=['algorithm', 'scenario'])['RFI'].agg(['mean', 'count', 'std'])
stats_rfi = my_ci_95(stats_rfi, r=5)
print(">>>>>>>>>>>>>>>>>>>> RFI <<<<<<<<<<<<<<<<<<<<\n", stats_rfi[['algorithm', 'scenario', 'mean', 'alpha']], '\n')

## Fragmentation metrics - counter
stats_rfc = dt_server_edge.groupby(by=['algorithm', 'scenario'])['rfc'].agg(['mean', 'count', 'std'])
stats_rfc = my_ci_95(stats_rfc)
print(">>>>>>>>>>>>>>>>>>>> Fragmentation Counter <<<<<<<<<<<<<<<<<<<<\n", stats_rfc[['algorithm', 'scenario', 'mean', 'alpha']], '\n')

## Fragmentation metrics - cpu
stats_rfcpu = dt_server_edge.groupby(by=['algorithm', 'scenario'])['rf_cpu'].agg(['mean', 'count', 'std'])
stats_rfcpu = my_ci_95(stats_rfcpu)
print(">>>>>>>>>>>>>>>>>>>> CPU Fragmentation <<<<<<<<<<<<<<<<<<<<\n", stats_rfcpu[['algorithm', 'scenario', 'mean', 'alpha']], '\n')

## Fragmentation metrics - ram
stats_rfram = dt_server_edge.groupby(by=['algorithm', 'scenario'])['rf_memory'].agg(['mean', 'count', 'std'])
stats_rfram = my_ci_95(stats_rfram)
print(">>>>>>>>>>>>>>>>>>>> RAM Fragmentation <<<<<<<<<<<<<<<<<<<<\n", stats_rfram[['algorithm', 'scenario', 'mean', 'alpha']], '\n')

## RII
stats_rii = dt_server_edge.groupby(by=['algorithm', 'scenario'])['RII'].agg(['mean', 'count', 'std'])
stats_rii = my_ci_95(stats_rii)
print(">>>>>>>>>>>>>>>>>>>> RII <<<<<<<<<<<<<<<<<<<<\n", stats_rii[['algorithm', 'scenario', 'mean', 'alpha']], '\n')

## Percent Server Access
stats_access = dt_server_access_avg.groupby(by=['algorithm', 'scenario', 'Object'])['weight'].agg(['mean', 'count', 'std'])
stats_access = my_ci_95(stats_access,  r=2)
stats_access['mean_percent'] = stats_access['mean']*100
print(">>>>>>>>>>>>>>>>>>>> Server Access <<<<<<<<<<<<<<<<<<<<\n", stats_access[['algorithm', 'scenario', 'Object', 'mean_percent']], '\n')

>>>>>>>>>>>>>>>>>>>> cumulative_sla_violations <<<<<<<<<<<<<<<<<<<<
   algorithm      scenario   mean  alpha
0    tetris  2ec_high_off   19.0   2.86
1    tetris   2ec_high_on   22.3   3.74
2    tetris   2ec_low_off   21.4   4.33
3    tetris    2ec_low_on   19.7   4.35
4      thea  2ec_high_off  225.2  27.50
5      thea   2ec_high_on   27.3   2.43
6      thea   2ec_low_off   32.7   4.81
7      thea    2ec_low_on   34.9   5.34 

>>>>>>>>>>>>>>>>>>>> drop <<<<<<<<<<<<<<<<<<<<
   algorithm      scenario  mean  alpha
0    tetris  2ec_high_off   0.0   0.00
1    tetris   2ec_high_on   0.0   0.00
2    tetris   2ec_low_off   0.0   0.00
3    tetris    2ec_low_on   0.0   0.00
4      thea  2ec_high_off   1.6   0.32
5      thea   2ec_high_on   0.0   0.00
6      thea   2ec_low_off   0.0   0.00
7      thea    2ec_low_on   0.0   0.00 

>>>>>>>>>>>>>>>>>>>> Delays <<<<<<<<<<<<<<<<<<<<
   algorithm      scenario    mean  alpha
0    tetris  2ec_high_off   30.78   1.27
1    tetris   2ec_high_on   76.30  1

In [7]:
df = dt_user.groupby(by=['algorithm', 'scenario', 'seed'])['cumulative_sla_violations'].max().reset_index()
print(">>>>> Latency SLA Violations <<<<<")
for s in scenario_list:
    df_s = df.loc[df['scenario'] == s].reset_index(drop=True)

    # Perform one-way ANOVA on SLA violations grouped by algorithm
    grouped_data = [df_s[df_s['algorithm'] == alg]['cumulative_sla_violations'].values for alg in df_s['algorithm'].unique()]
    f_statistic, p_value = stats.f_oneway(*grouped_data)

    # Print the results
    print(s, "One-way ANOVA results:")
    print(f"F-statistic: {f_statistic:.4f}")
    print(f"P-value: {p_value:.4f}")

df = dt_user_drop.groupby(by=['algorithm', 'scenario', 'seed'])['drop'].sum().reset_index()
print("\n>>>>> Drop SLA Violations <<<<<")
for s in scenario_list:
    df_s = df.loc[df['scenario'] == s].reset_index(drop=True)

    # Perform one-way ANOVA on SLA violations grouped by algorithm
    grouped_data = [df_s[df_s['algorithm'] == alg]['drop'].values for alg in df_s['algorithm'].unique()]
    f_statistic, p_value = stats.f_oneway(*grouped_data)

    # Print the results
    print(s, "One-way ANOVA results:")
    print(f"F-statistic: {f_statistic:.4f}")
    print(f"P-value: {p_value:.8f}")

df = dt_user.groupby(by=['algorithm', 'scenario', 'seed'])['Delays'].mean().reset_index()
print("\n>>>>> Average Latency <<<<<")
for s in scenario_list:
    df_s = df.loc[df['scenario'] == s].reset_index(drop=True)

    # Perform one-way ANOVA on SLA violations grouped by algorithm
    grouped_data = [df_s[df_s['algorithm'] == alg]['Delays'].values for alg in df_s['algorithm'].unique()]
    f_statistic, p_value = stats.f_oneway(*grouped_data)

    # Print the results
    print(s, "One-way ANOVA results:")
    print(f"F-statistic: {f_statistic:.4f}")
    print(f"P-value: {p_value:.4f}")

df = dt_power.groupby(by=['algorithm', 'scenario', 'seed'])['Power Consumption'].mean().reset_index()
print("\n>>>>> Power Consumption <<<<<")
for s in scenario_list:
    df_s = df.loc[df['scenario'] == s].reset_index(drop=True)

    # Perform one-way ANOVA on SLA violations grouped by algorithm
    grouped_data = [df_s[df_s['algorithm'] == alg]['Power Consumption'].values for alg in df_s['algorithm'].unique()]
    f_statistic, p_value = stats.f_oneway(*grouped_data)

    # Print the results
    print(s, "One-way ANOVA results:")
    print(f"F-statistic: {f_statistic:.4f}")
    print(f"P-value: {p_value:.4f}")

df = dt_server_edge.groupby(by=['algorithm', 'scenario', 'seed'])['RFI'].mean().reset_index()
print("\n>>>>> RFI <<<<<")
for s in scenario_list:
    df_s = df.loc[df['scenario'] == s].reset_index(drop=True)

    # Perform one-way ANOVA on SLA violations grouped by algorithm
    grouped_data = [df_s[df_s['algorithm'] == alg]['RFI'].values for alg in df_s['algorithm'].unique()]
    f_statistic, p_value = stats.f_oneway(*grouped_data)

    # Print the results
    print(s, "One-way ANOVA results:")
    print(f"F-statistic: {f_statistic:.4f}")
    print(f"P-value: {p_value:.4f}")

>>>>> Latency SLA Violations <<<<<
2ec_low_on One-way ANOVA results:
F-statistic: 18.6825
P-value: 0.0004
2ec_low_off One-way ANOVA results:
F-statistic: 11.7207
P-value: 0.0030
2ec_high_on One-way ANOVA results:
F-statistic: 4.8263
P-value: 0.0414
2ec_high_off One-way ANOVA results:
F-statistic: 213.6653
P-value: 0.0000

>>>>> Drop SLA Violations <<<<<
2ec_low_on One-way ANOVA results:
F-statistic: nan
P-value: nan
2ec_low_off One-way ANOVA results:
F-statistic: nan
P-value: nan
2ec_high_on One-way ANOVA results:
F-statistic: nan
P-value: nan
2ec_high_off One-way ANOVA results:
F-statistic: 96.0000
P-value: 0.00000001

>>>>> Average Latency <<<<<
2ec_low_on One-way ANOVA results:
F-statistic: 148.6724
P-value: 0.0000
2ec_low_off One-way ANOVA results:
F-statistic: 8.9672
P-value: 0.0078
2ec_high_on One-way ANOVA results:
F-statistic: 839.7437
P-value: 0.0000
2ec_high_off One-way ANOVA results:
F-statistic: 3.9552
P-value: 0.0621

>>>>> Power Consumption <<<<<
2ec_low_on One-way ANOVA 

c:\Users\lucasa88\.conda\envs\smartplan\Lib\site-packages\scipy\stats\_axis_nan_policy.py:586: ConstantInputWarning: Each of the input arrays is constant; the F statistic is not defined or infinite
  res = hypotest_fun_out(*samples, **kwds)


In [8]:
# RESULTS TO USE ON OVERLEAF PLOTS

## SLA Violations
stats_sla = dt_user.groupby(by=['algorithm', 'scenario', 'seed'])['cumulative_sla_violations'].max().groupby(by=['algorithm']).agg(['mean', 'count', 'std'])
stats_sla = my_ci_95(stats_sla)
print(">>>>>>>>>>>>>>>>>>>> cumulative_sla_violations <<<<<<<<<<<<<<<<<<<<\n", stats_sla[['algorithm', 'mean', 'alpha']], '\n')

## Drop
stats_drop = dt_user_drop.groupby(by=['algorithm', 'scenario', 'seed'])['drop'].sum().groupby(by=['algorithm']).agg(['mean', 'count', 'std'])
stats_drop = my_ci_95(stats_drop)
print(">>>>>>>>>>>>>>>>>>>> drop <<<<<<<<<<<<<<<<<<<<\n", stats_drop[['algorithm', 'mean', 'alpha']], '\n')

## Delay
stats_delay = dt_user.groupby(by=['algorithm', 'scenario', 'seed'])['Delays'].mean().groupby(by=['algorithm']).agg(['mean', 'count', 'std'])
stats_delay = my_ci_95(stats_delay)
print(">>>>>>>>>>>>>>>>>>>> Delays <<<<<<<<<<<<<<<<<<<<\n", stats_delay[['algorithm', 'mean', 'alpha']], '\n')

## Power Consumption
stats_power = dt_power.groupby(by=['algorithm', 'scenario', 'seed'])['Power Consumption'].mean().groupby(by=['algorithm']).agg(['mean', 'count', 'std'])
stats_power = my_ci_95(stats_power)
print(">>>>>>>>>>>>>>>>>>>> Power <<<<<<<<<<<<<<<<<<<<\n", stats_power[['algorithm', 'mean', 'alpha']], '\n')

## RFI
stats_rfi = dt_server_edge.groupby(by=['algorithm', 'scenario', 'seed'])['RFI'].mean().groupby(by=['algorithm']).agg(['mean', 'count', 'std'])
stats_rfi = my_ci_95(stats_rfi, r=5)
print(">>>>>>>>>>>>>>>>>>>> RFI <<<<<<<<<<<<<<<<<<<<\n", stats_rfi[['algorithm', 'mean', 'alpha']], '\n')

## Percent Server Access
dt_server_access_avg_algo = dt_server_access_avg.groupby(by=['algorithm', 'Object'])['weight'].sum().reset_index()
stats_access = dt_server_access_avg_algo.merge(dt_server_access_avg_algo.groupby(by=['algorithm'])['weight'].sum().reset_index().rename(columns={'weight':'total_weight'}), how='left', on='algorithm')
stats_access['mean_percent'] = (stats_access['weight']/stats_access['total_weight'])*100
print(">>>>>>>>>>>>>>>>>>>> Server Access <<<<<<<<<<<<<<<<<<<<\n", stats_access[['algorithm', 'Object', 'mean_percent']], '\n')

>>>>>>>>>>>>>>>>>>>> cumulative_sla_violations <<<<<<<<<<<<<<<<<<<<
   algorithm   mean  alpha
0    tetris  20.60    1.9
1      thea  80.03   27.2 

>>>>>>>>>>>>>>>>>>>> drop <<<<<<<<<<<<<<<<<<<<
   algorithm  mean  alpha
0    tetris   0.0   0.00
1      thea   0.4   0.23 

>>>>>>>>>>>>>>>>>>>> Delays <<<<<<<<<<<<<<<<<<<<
   algorithm   mean  alpha
0    tetris  41.16   6.41
1      thea  75.59  16.70 

>>>>>>>>>>>>>>>>>>>> Power <<<<<<<<<<<<<<<<<<<<
   algorithm    mean  alpha
0    tetris  185.20   4.90
1      thea  194.15   3.92 

>>>>>>>>>>>>>>>>>>>> RFI <<<<<<<<<<<<<<<<<<<<
   algorithm     mean    alpha
0    tetris  0.20781  0.01556
1      thea  0.43743  0.05169 

>>>>>>>>>>>>>>>>>>>> Server Access <<<<<<<<<<<<<<<<<<<<
    algorithm        Object  mean_percent
0     tetris  EdgeServer_1     16.809439
1     tetris  EdgeServer_2     16.164351
2     tetris  EdgeServer_3     17.449529
3     tetris  EdgeServer_4     17.136462
4     tetris  EdgeServer_5     19.998892
5     tetris  EdgeServ

In [9]:
# RESULTS TO USE ON OVERLEAF PLOTS


## SLA Violations
stats_sla = dt_user.groupby(by=['cloud', 'scenario', 'seed'])['cumulative_sla_violations'].max().groupby(by=['cloud']).agg(['mean', 'count', 'std'])
stats_sla = my_ci_95(stats_sla)
print(">>>>>>>>>>>>>>>>>>>> cumulative_sla_violations <<<<<<<<<<<<<<<<<<<<\n", stats_sla[['cloud', 'mean', 'alpha']], '\n')

## Drop
stats_drop = dt_user_drop.groupby(by=['cloud', 'scenario', 'seed'])['drop'].sum().groupby(by=['cloud']).agg(['mean', 'count', 'std'])
stats_drop = my_ci_95(stats_drop)
print(">>>>>>>>>>>>>>>>>>>> drop <<<<<<<<<<<<<<<<<<<<\n", stats_drop[['cloud', 'mean', 'alpha']], '\n')

## Delay
stats_delay = dt_user.groupby(by=['cloud', 'scenario', 'seed'])['Delays'].mean().groupby(by=['cloud']).agg(['mean', 'count', 'std'])
stats_delay = my_ci_95(stats_delay)
print(">>>>>>>>>>>>>>>>>>>> Delays <<<<<<<<<<<<<<<<<<<<\n", stats_delay[['cloud', 'mean', 'alpha']], '\n')

## Power Consumption
stats_power = dt_power.groupby(by=['cloud', 'scenario', 'seed'])['Power Consumption'].mean().groupby(by=['cloud']).agg(['mean', 'count', 'std'])
stats_power = my_ci_95(stats_power)
print(">>>>>>>>>>>>>>>>>>>> Power <<<<<<<<<<<<<<<<<<<<\n", stats_power[['cloud', 'mean', 'alpha']], '\n')

## RFI
stats_rfi = dt_server_edge.groupby(by=['cloud', 'scenario', 'seed'])['RFI'].mean().groupby(by=['cloud']).agg(['mean', 'count', 'std'])
stats_rfi = my_ci_95(stats_rfi, r=5)
print(">>>>>>>>>>>>>>>>>>>> RFI <<<<<<<<<<<<<<<<<<<<\n", stats_rfi[['cloud', 'mean', 'alpha']], '\n')

## Percent Server Access
dt_server_access_avg_cloud = dt_server_access_avg.groupby(by=['cloud', 'Object'])['weight'].sum().reset_index()
stats_access = dt_server_access_avg_cloud.merge(dt_server_access_avg_cloud.groupby(by=['cloud'])['weight'].sum().reset_index().rename(columns={'weight':'total_weight'}), how='left', on='cloud')
stats_access['mean_percent'] = (stats_access['weight']/stats_access['total_weight'])*100
print(">>>>>>>>>>>>>>>>>>>> Server Access <<<<<<<<<<<<<<<<<<<<\n", stats_access[['cloud', 'Object', 'mean_percent']], '\n')

>>>>>>>>>>>>>>>>>>>> cumulative_sla_violations <<<<<<<<<<<<<<<<<<<<
   cloud   mean  alpha
0   off  129.2  45.25
1    on   31.3   3.28 

>>>>>>>>>>>>>>>>>>>> drop <<<<<<<<<<<<<<<<<<<<
   cloud  mean  alpha
0   off   0.8   0.39
1    on   0.0   0.00 

>>>>>>>>>>>>>>>>>>>> Delays <<<<<<<<<<<<<<<<<<<<
   cloud   mean  alpha
0   off  28.71   0.98
1    on  93.25  11.82 

>>>>>>>>>>>>>>>>>>>> Power <<<<<<<<<<<<<<<<<<<<
   cloud    mean  alpha
0   off  193.28   4.15
1    on  184.99   2.47 

>>>>>>>>>>>>>>>>>>>> RFI <<<<<<<<<<<<<<<<<<<<
   cloud     mean    alpha
0   off  0.46990  0.03081
1    on  0.25406  0.02962 

>>>>>>>>>>>>>>>>>>>> Server Access <<<<<<<<<<<<<<<<<<<<
    cloud        Object  mean_percent
0    off  EdgeServer_1     17.062375
1    off  EdgeServer_2     16.872414
2    off  EdgeServer_3     15.686290
3    off  EdgeServer_4     17.351452
4    off  EdgeServer_5     19.186654
5    off  EdgeServer_6     13.840814
6     on  EdgeServer_1     11.772199
7     on  EdgeServer_2     15.55